 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [3]:
from matplotlib.style import available

perplex_source_list_re = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')

relinker = lpz.ZoteroLinkConverter()

def collect_and_fix_body_links(file_text: str, verbose: bool = False) -> tuple[str, pd.DataFrame]:
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    

    section_parts = file_text.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts

    # Reassign body cite numbers if duplicate URLs are found in the sources
    source_matches = list(perplex_source_list_re.finditer(citations))
    if len(source_matches) < 1:
        print('Found no sources in file_text')
        
    source_citenum_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]
    citenums_to_url = relinker.citenums_to_urls_dedup(source_citenum_url_pairs, verbose=verbose)
        
    body_dedup = relinker.replace_body_citenums(body, citenums_to_url.new_num.to_dict())
    
    return body_dedup, citenums_to_url

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    file_text = perplexity_file.read_text(encoding='utf-8')
    
    body_dedup, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(body_dedup, citenums_to_url)
    
    body_relinked = rfw.heirarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)
    relinked_file.write_text(f'\n*{source_link}*\n# Response\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', encoding='utf-8')

NameError: name 'lpz' is not defined

In [ ]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_perplexity_export(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
Done.


### Test merging

In [4]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

#chat_files = list(datdir.glob('*.md'))
chat_files = [perplexity_dialog_file]

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [ ]:
verbose = False
num_chat_files = len(chat_files)
all_bodies, all_citenums_to_url = [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'{chat_file.stem}')

    file_text = chat_file.read_text(encoding='utf-8')
    
    body_dedup, citenums_to_url = collect_and_fix_body_links(file_text, verbose=verbose)
    all_bodies.append(body_dedup)

    citenums_to_url[['file_index','chat_file']] = file_index, chat_file
    all_citenums_to_url.append(citenums_to_url.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} citation numbers')

#### Make a unified cite number set for the merged document

In [6]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['new_num_int'] = df['new_num'].astype(int)

grouped = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_new_num_int=('new_num_int', 'mean')
).reset_index()

grouped = grouped.sort_values(by=['mean_file_index', 'mean_new_num_int'], 
                              ascending=True).reset_index(drop=True)

grouped['citenum_merged'] = np.arange(1, len(grouped) + 1).astype(str) # citenum == rank as sttring

# Merge back the new citenumes
df = df.merge(grouped[['url', 'citenum_merged']], on='url')

In [ ]:
all_citenums_to_url = (df.sort_values('citenum_merged')
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('new_num_int', axis=1)
                       .set_index('file_index'))

# all_citenums_to_url

#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [ ]:
# make a single mapping from unified citenums to urls
unified_citenums = all_citenums_to_url[['new_num', 'url']].drop_duplicates()
unified_citenums.index = unified_citenums['new_num']
unified_citenums.index.name = 'orig_num' # match expectations below TODO: needed?

response_heading = 'Responses' if len(chat_files) > 1 else 'Response'
body_unified = 
bodies_unified, chat_source_file_link = [], []
for file_index, body_dedup in enumerate(all_bodies):
    citenums_to_url_this = all_citenums_to_url.loc[file_index]
    citenums_dedup_to_unified = citenums_to_url_this.set_index('doc_dedup_num').new_num.to_dict()

    bodies_unified[file_index] = relinker.replace_body_citenums(body_dedup, citenums_dedup_to_unified)
    chat_source_file_link[file_index] = rfw.file_link_md('chat_source', chat_files[file_index])

##### Insert links to Obsidian or Zotero

In [ ]:


unified_body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(unified_body , unified_citenums)

unified_body_relinked = rfw.hierarch_shift_markdown_headers(unified_body_relinked, top_level=2)
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group(1))))

ic(datdir, merged_output_file)
if num_chat_files > 1:
    unified_body += f'# {chat_files[file_index].name}\n*{source_link}*\n'
else:
    unified_body = f'*{source_link}*\n{#Response}}'
    
unified_body += f'{body_unified_this}\n'
response = 'Responses' if len(chat_files) > 1 else 'Response'
merged_output_file.write_text(f'# {response}\n{unified_body_relinked}\n# Citations\n{relinked_sources}', encoding='utf-8')

print("Done.")

SyntaxError: invalid syntax (4023139670.py, line 3)

In [ ]:
import re
from datetime import datetime

def is_save_my_chatbot_file(file_path):
    """
    Checks if a given Markdown file is from the SaveMyChatbot plugin.

    Args:
        file_path (str): Path to the Markdown file.

    Returns:
        bool: True if the file matches the SaveMyChatbot format, False otherwise.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Read the first two lines of the file
            heading = file.readline().strip()
            metadata = file.readline().strip()
        
        # Line 1: Check if it starts with a heading (e.g., "# What is...")
        if not heading.startswith("# "):
            return False
        
        # Line 2: Check for "Exported on" followed by a date and time
        exported_pattern = r"Exported on (\d{2}/\d{2}/\d{4}) at (\d{2}:\d{2}:\d{2})"
        match = re.search(exported_pattern, metadata)
        if not match:
            return False
        
        # Validate date and time format
        try:
            datetime.strptime(f"{match.group(1)} {match.group(2)}", "%d/%m/%Y %H:%M:%S")
        except ValueError:
            return False
        
        # Check for links to Perplexity.ai and SaveMyChatbot in line 2
        if not ("Perplexity.ai" in metadata and "SaveMyChatbot" in metadata):
            return False
        
        return True

    except Exception as e:
        print(f"Error reading file: {e}")
        return False

# Example usage
file_path = "example.md"
if is_save_my_chatbot_file(file_path):
    print("The file is from SaveMyChatbot.")
else:
    print("The file is NOT from SaveMyChatbot.")


Error reading file: [Errno 2] No such file or directory: 'example.md'
The file is NOT from SaveMyChatbot.
